In [1]:
import pandas as pd

In [4]:
df = pd.read_csv(r"D:\Credit card project\data\raw\credit_card_transaction_flow.csv")

In [5]:
df

,Customer ID,Name,Surname,Gender,Birthdate,Transaction Amount,Date,Merchant Name,Category
0,752858,Sean,Rodriguez,F,20-10-2002,35.47,03-04-2023,Smith-Russell,Cosmetic
1,26381,Michelle,Phelps,NaN,24-10-1985,2552.72,17-07-2023,"Peck, Spence and Young",Travel
2,305449,Jacob,Williams,M,25-10-1981,115.97,20-09-2023,Steele Inc,Clothing
3,988259,Nathan,Snyder,M,26-10-1977,11.31,11-01-2023,"Wilson, Wilson and Russell",Cosmetic
4,764762,Crystal,Knapp,F,02-11-1951,62.21,13-06-2023,Palmer-Hinton,Electronics
...,...,...,...,...,...,...,...,...,...
49995,891845,Christine,Leach,F,21-10-1997,108.74,30-08-2023,Alexander Ltd,Market
49996,800560,Anna,Allen,F,21-10-1999,133.66,03-05-2023,Knapp-Calhoun,Cosmetic
49997,133285,Nicole,Franklin,M,26-10-1979,464.29,12-02-2023,"Cantrell, Haynes and Ballard",Market
49998,616122,Maria,Keller,M,25-10-1981,472.57,25-03-2023,"Wilson, Jackson and Beard",Market


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Customer ID         50000 non-null  int64  
 1   Name                50000 non-null  str    
 2   Surname             50000 non-null  str    
 3   Gender              44953 non-null  str    
 4   Birthdate           50000 non-null  str    
 5   Transaction Amount  50000 non-null  float64
 6   Date                50000 non-null  str    
 7   Merchant Name       50000 non-null  str    
 8   Category            50000 non-null  str    
dtypes: float64(1), int64(1), str(7)
memory usage: 3.4 MB


In [8]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")


In [9]:
df.columns

Index(['customer_id', 'name', 'surname', 'gender', 'birthdate',
       'transaction_amount', 'date', 'merchant_name', 'category'],
      dtype='str')

In [10]:
df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y")
df["birthdate"] = pd.to_datetime(df["birthdate"], format="%d-%m-%Y")

In [11]:
df["gender"] = df["gender"].fillna("Unknown")

In [12]:
df["month"] = df["date"].dt.to_period("M").astype(str)

In [13]:
df["month"]

0        2023-04
1        2023-07
2        2023-09
3        2023-01
4        2023-06
          ...   
49995    2023-08
49996    2023-05
49997    2023-02
49998    2023-03
49999    2023-05
Name: month, Length: 50000, dtype: str

In [15]:
df["weekday"] = df["date"].dt.day_name()

In [16]:
df["is_weekend"]

0        False
1        False
2        False
3        False
4        False
         ...  
49995    False
49996    False
49997     True
49998     True
49999     True
Name: is_weekend, Length: 50000, dtype: bool

In [17]:
df["age"] = (df["date"] - df["birthdate"]).dt.days // 365
df["age_group"] = pd.cut(df["age"], bins=[0, 25, 35, 50, 100],
                         labels=["Under 26", "26-35", "36-50", "50+"])

In [18]:
df["age_group"]

0        Under 26
1           36-50
2           36-50
3           36-50
4             50+
           ...   
49995    Under 26
49996    Under 26
49997       36-50
49998       36-50
49999         50+
Name: age_group, Length: 50000, dtype: category
Categories (4, str): ['Under 26' < '26-35' < '36-50' < '50+']

In [22]:
df["high_value"] = df["transaction_amount"] >= df.groupby("category")["transaction_amount"].transform(lambda s: s.quantile(0.95))
df.to_csv("clean_transactions.csv", index=False)

In [20]:
print(df["age_group"].isnull().sum())
print(df["gender"].value_counts())
print(df["date"].min(), df["date"].max())
print(df["is_weekend"].value_counts())
print(df.groupby("category")["transaction_amount"].agg(["count", "sum", "mean"]).round(2))

0
gender
F          22713
M          22240
Unknown     5047
Name: count, dtype: int64
2023-01-01 00:00:00 2023-10-14 00:00:00
is_weekend
False    35731
True     14269
Name: count, dtype: int64
             count          sum     mean
category                                
Clothing      8261   1319342.47   159.71
Cosmetic      8243    876672.44   106.35
Electronics   8324   4394092.10   527.88
Market        8382   2151134.42   256.64
Restaurant    8413    464488.60    55.21
Travel        8377  12900231.94  1539.96


In [21]:
df.to_csv("clean_transactions.csv", index=False)
print("Done!", df.shape)

Done! (50000, 15)
